In [ ]:
from nba_api.stats.endpoints import boxscoresummaryv2

# Game ID: 0042200101 (Lakers vs Nuggets, May 16, 2023)
game_id = "0042200101"

# Fetch data
box_score = boxscoresummaryv2.BoxScoreSummaryV2(game_id=game_id)
inactive_players = box_score.inactive_players.get_data_frame()

print("--- INACTIVE PLAYERS ---")
# Show just the relevant columns
print(inactive_players[["PLAYER_ID", "FIRST_NAME", "LAST_NAME", "TEAM_ABBREVIATION"]])

In [ ]:
from nba_api.stats.endpoints import commonteamroster

# Team ID for Denver Nuggets: 1610612743
team_id = "1610612743"

# Fetch Roster
roster = commonteamroster.CommonTeamRoster(team_id=team_id, season="2022-23")
df_roster = roster.common_team_roster.get_data_frame()

print("--- TEAM ROSTER & POSITIONS ---")
print(df_roster[["PLAYER", "POSITION", "NUM"]].head(10))

In [ ]:
from nba_api.stats.endpoints import winprobabilitypbp

# Same Game ID as above
game_id = "0042200101"

# Fetch Win Probability Data
win_prob = winprobabilitypbp.WinProbabilityPBP(game_id=game_id)
df_pbp = win_prob.game_win_probability.get_data_frame()

# Filter for a "Garbage Time" scenario (e.g., 4th Quarter, Margin > 15)
garbage_time_plays = df_pbp[(df_pbp["PERIOD"] == 4) & (df_pbp["HOME_SCORE_MARGIN"].abs() > 15)]

print("--- GARBAGE TIME FRAMES ---")
if not garbage_time_plays.empty:
    print(
        garbage_time_plays[["PERIOD", "SECONDS_REMAINING", "HOME_SCORE_MARGIN", "HOME_PCT"]].head()
    )
else:
    print("No garbage time detected in this game based on criteria.")

In [ ]:
from nba_api.stats.endpoints import playbyplayv2

# 1. Fetch Play-by-Play Data for a specific game
# Example: Lakers vs Warriors (Season 2022-23)
pbp = playbyplayv2.PlayByPlayV2(game_id="0022200001")
df = pbp.get_data_frames()[0]

# 2. Filter to relevant columns
# PCTIMESTRING = Clock time (e.g., "11:45")
# SCOREMARGIN = The lead size (e.g., "5", "-2", "TIE")
cols = ["PERIOD", "PCTIMESTRING", "SCORE", "SCOREMARGIN", "HOMEDESCRIPTION", "VISITORDESCRIPTION"]
df = df[cols].copy()

# 3. Data Cleaning (Crucial Step)
# SCOREMARGIN is often 'None' for non-scoring events (rebounds, fouls).
# We forward-fill it so every event "knows" the current score.
df["SCOREMARGIN"] = df["SCOREMARGIN"].ffill()

# Handle "TIE" values (convert to 0) and missing start values
df["SCOREMARGIN"] = df["SCOREMARGIN"].replace("TIE", 0).fillna(0)

# Convert to integer for ML models
df["SCOREMARGIN"] = df["SCOREMARGIN"].astype(int)


# 4. Create a "Time Remaining" Feature (Optional but recommended)
# Convert string time "11:45" to total seconds remaining in game
def time_to_seconds(time_str, period):
    minutes, seconds = map(int, time_str.split(":"))
    period_seconds = (minutes * 60) + seconds

    # Regular quarters are 12 mins (720s). OT is 5 mins (300s).
    if period <= 4:
        return (4 - period) * 720 + period_seconds
    else:
        return period_seconds  # OT logic varies, but this is the base idea


df["SECONDS_REMAINING"] = df.apply(
    lambda x: time_to_seconds(x["PCTIMESTRING"], x["PERIOD"]), axis=1
)

# Preview the clean data for your model
print(df[["PERIOD", "PCTIMESTRING", "SCORE", "SCOREMARGIN", "SECONDS_REMAINING"]].head(10))

In [2]:
from nba_api.stats.endpoints import playbyplayv3

# 1. Fetch Play-by-Play Data for a specific game
# Example: Lakers vs Warriors (Season 2022-23)
pbp = playbyplayv3.PlayByPlayV3(game_id="0022200001")
df = pbp.get_data_frames()[0]
df

,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,...,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId
0,0022200001,2,PT12M00.00S,1,0,,0,,,0,...,0,0,0,,Start of 1st Period (7:36 PM EST),period,start,0,0,1
1,0022200001,4,PT12M00.00S,1,1610612738,BOS,201143,Horford,A. Horford,0,...,,,0,h,Jump Ball Horford vs. Embiid: Tip to Harris,Jump Ball,,1,0,2
2,0022200001,7,PT11M38.00S,1,1610612755,PHI,203954,Embiid,J. Embiid,-118,...,,,0,v,MISS Embiid 13' Turnaround Fadeaway Shot,Missed Shot,Turnaround Fadeaway shot,1,2,3
3,0022200001,7,PT11M38.00S,1,1610612738,BOS,1627759,Brown,J. Brown,0,...,,,0,h,Brown BLOCK (1 BLK),,,1,2,4
4,0022200001,9,PT11M35.00S,1,1610612755,PHI,200782,Tucker,P. Tucker,0,...,,,0,v,Tucker REBOUND (Off:1 Def:0),Rebound,Unknown,1,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
463,0022200001,637,PT00M32.10S,4,1610612755,PHI,202699,Harris,T. Harris,0,...,,,0,v,Harris STEAL (3 STL),,,1,0,464
464,0022200001,639,PT00M29.80S,4,1610612755,PHI,202699,Harris,T. Harris,0,...,126,117,243,v,Harris 1' Running Dunk (18 PTS),Made Shot,Running Dunk Shot,1,2,465
465,0022200001,640,PT00M06.90S,4,1610612738,BOS,201143,Horford,A. Horford,131,...,,,0,h,MISS Horford 25' 3PT Jump Shot,Missed Shot,Jump Shot,1,3,466
466,0022200001,641,PT00M04.20S,4,1610612755,PHI,201935,Harden,J. Harden,0,...,,,0,v,Harden REBOUND (Off:0 Def:8),Rebound,Unknown,1,0,467
